In [53]:
from collections import defaultdict
from lenskit.algorithms.als import BiasedMF
import numpy as np
import pandas as pd

import utils

# CF Example
This notebook shows how the `BiasedMF` model from `LensKit` can be used to generate predicted scores using the matrix factorization technique of collaborative filtering (CF).

In [42]:
# test data to use in this example
test_data = pd.DataFrame({"user": ["Kurt","Kurt","Kurt","Kurt","Lars","Lars","Lars","Lars","Brian","Brian","Brian","Brian"],
                          "item": [1,4,6,7,2,8,4,9,3,5,6,7],
                          "rating": [0.4,0.3,0.2,0.2,0.4,0.5,0.9,0.1,0.1,0.4,0.3,0.3]})

In [43]:
# lists of unique users and items
users = [1,2,3]
items = [1,2,3,4,5,6,7,8,9]

Using `predict_for_user` to generate predicted scores per user in a for loop (slow method):

In [44]:
scores_dict = {}

# initializing the BiasedMF model
mf = BiasedMF(features=100, 
            iterations=10, 
            reg=0.1, 
            damping=5, 
            bias=True,
            rng_spec=42)

# fitting the model
mf.fit(test_data)

# generating scores for each user and saving to the scores_dict
for user in users:
    scores = list(mf.predict_for_user(user, items))
    scores_dict[user] = scores

print(scores_dict)

{1: [nan, nan, nan, nan, nan, nan, nan, nan, nan], 2: [nan, nan, nan, nan, nan, nan, nan, nan, nan], 3: [nan, nan, nan, nan, nan, nan, nan, nan, nan]}


Computing the matrix product UV and adding the bias terms to obtain the predicted ratings in a more efficient way:

In [45]:
# U: [n_users × k]
U = mf.user_features_

# V: [n_items × k]
V = mf.item_features_

# biases:
ub = mf.bias.user_offsets_.reindex(mf.user_index_).to_numpy()
ib = mf.bias.item_offsets_.reindex(mf.item_index_).to_numpy()
mu = mf.bias.mean_
ub = ub.reshape(-1, 1)   # shape [n_users, 1]
ib = ib.reshape(1, -1)   # shape [1, n_items]

# full prediction matrix: [n_users × n_items]
pred_matrix = U @ V.T + ub + ib + mu
print(pred_matrix)

[[0.33078546 0.33096144 0.27882089 0.3964864  0.33179581 0.29518879
  0.29518879 0.34800007 0.27984554]
 [0.32299736 0.31864834 0.26873689 0.34407923 0.31813069 0.27994017
  0.27994017 0.32528416 0.29874091]
 [0.37901356 0.40112873 0.35085222 0.6617978  0.40369493 0.37942909
  0.37942909 0.46880599 0.19809696]]


The `.fit` method trains the `BiasedMF` model for the number of epochs specified by the `iterations` hyperparameter in one go. To train incrementally while evaluating on the validation set per epoch (and possibly apply early stopping) use `fit_iters` instead:

In [46]:
n_epochs = 10

mf = BiasedMF(features=100, iterations=n_epochs, bias=True)

# create the epoch generator
epoch_gen = mf.fit_iters(test_data)

for epoch in range(n_epochs):
    print("Running epoch:", epoch+1)

    # run an epoch
    next(epoch_gen)

    # U: [n_users × k]
    U = mf.user_features_

    # V: [n_items × k]
    V = mf.item_features_

    # biases:
    ub = mf.bias.user_offsets_.reindex(mf.user_index_).to_numpy()
    ib = mf.bias.item_offsets_.reindex(mf.item_index_).to_numpy()
    mu = mf.bias.mean_
    ub = ub.reshape(-1, 1)   # shape [n_users, 1]
    ib = ib.reshape(1, -1)   # shape [1, n_items]
    
    # full prediction matrix: [n_users × n_items]
    pred_matrix = U @ V.T + ub + ib + mu
    print(pred_matrix)

Running epoch: 1
[[0.33205375 0.33070862 0.24849829 0.38819975 0.34334429 0.29503179
  0.29504564 0.3458133  0.28600602]
 [0.3391441  0.31787495 0.26584259 0.34029411 0.31923063 0.27067891
  0.27070492 0.32542564 0.29608726]
 [0.38314038 0.40424981 0.36156125 0.67638684 0.3996147  0.37800232
  0.37797723 0.47450764 0.19342701]]
Running epoch: 2
[[0.33140011 0.33102644 0.26426775 0.39080554 0.33734654 0.295089
  0.295089   0.34656674 0.28438397]
 [0.32405065 0.31869578 0.26737714 0.34474722 0.31864931 0.27939303
  0.27939304 0.32559647 0.29792263]
 [0.37992103 0.40084462 0.35862668 0.66239311 0.40072966 0.37892511
  0.3789251  0.46874614 0.19747212]]
Running epoch: 3
[[0.33125586 0.33100144 0.27017878 0.39208008 0.335092   0.29507188
  0.29507188 0.34688133 0.28336308]
 [0.32304838 0.31864221 0.26767557 0.34434734 0.31853548 0.27992738
  0.27992738 0.32535932 0.29849786]
 [0.37921815 0.40114569 0.35705506 0.66180406 0.4013291  0.37924132
  0.37924132 0.46881764 0.19809491]]
Running epoc

One should notice that the predicted ratings obtained after 10 epochs are identical to the ratings obtained above using `.fit`. Now let's try to extract the 

In [61]:
print(pred_matrix)
top_idx = np.argpartition(pred_matrix, -6, axis=1)[:, -6:]

# now sort those top-k indices by their values (descending) for each row
rows = np.arange(pred_matrix.shape[0])[:, None]
print(rows)
topk_idx_sorted = top_idx[rows, np.argsort(-pred_matrix[rows, top_idx], axis=1)]
print(topk_idx_sorted)

[[0.33087252 0.33096791 0.2788845  0.39577864 0.33177155 0.295143
  0.295143   0.34782122 0.28040797]
 [0.32299615 0.3186483  0.2685348  0.34408453 0.31820776 0.27994343
  0.27994343 0.32528549 0.2987367 ]
 [0.37901649 0.40112874 0.3518771  0.66179723 0.40330403 0.37941422
  0.37941422 0.46880586 0.19809736]]
[[0]
 [1]
 [2]]
[[3 7 4 1 0 6]
 [3 7 0 1 4 8]
 [3 7 4 1 5 6]]


In [51]:
mf.item_index_

Int64Index([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64', name='item')

In [49]:
mf.user_index_

Index(['Brian', 'Kurt', 'Lars'], dtype='object', name='user')

In [58]:
users = mf.user_index_
items = mf.item_index_
recs_dict = defaultdict(list)

for n, user in enumerate(users):
    recs_idx = topk_idx_sorted[n, :].tolist()
    recs = [items[rec_idx] for rec_idx in recs_idx]
    recs_dict[user] = recs

print(recs_dict)

defaultdict(<class 'list'>, {'Brian': [4, 8, 5, 2, 1, 7], 'Kurt': [4, 8, 1, 2, 5, 9], 'Lars': [4, 8, 5, 2, 6, 7]})
